In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv('./E-Commerce_Churn_Data.csv')

print("Dataset Shape:", df.shape)

print(df.head())

Dataset Shape: (5630, 20)
   CustomerID  Churn  Tenure PreferredLoginDevice  CityTier  WarehouseToHome  \
0       50001      1     4.0         Mobile Phone         3              6.0   
1       50002      1     NaN                Phone         1              8.0   
2       50003      1     NaN                Phone         1             30.0   
3       50004      1     0.0                Phone         3             15.0   
4       50005      1     0.0                Phone         1             12.0   

  PreferredPaymentMode  Gender  HourSpendOnApp  NumberOfDeviceRegistered  \
0           Debit Card  Female             3.0                         3   
1                  UPI    Male             3.0                         4   
2           Debit Card    Male             2.0                         4   
3           Debit Card    Male             2.0                         4   
4                   CC    Male             NaN                         3   

     PreferedOrderCat  SatisfactionS

In [2]:
# Summary of data types and memory usage
df.info()

# Check for duplicate rows
print("Duplicate Rows:", df.duplicated().sum())

# List column names to identify unique IDs or unwanted features
print("Columns:", df.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   str    
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   str    
 7   Gender                       5630 non-null   str    
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   str    
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   str    
 13  NumberOfAddress              

In [3]:
# Count missing values per column
missing_counts = df.isnull().sum()

# Filter and display only columns with missing values
missing_data = missing_counts[missing_counts > 0]
print("Missing Values Per Column:\n", missing_data)

# Percentage of missing data per column
missing_percentage = (missing_data / len(df)) * 100
print("\nPercentage Missing:\n", missing_percentage)

Missing Values Per Column:
 Tenure                         264
WarehouseToHome                251
HourSpendOnApp                 255
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderCount                     258
DaySinceLastOrder              307
dtype: int64

Percentage Missing:
 Tenure                         4.689165
WarehouseToHome                4.458259
HourSpendOnApp                 4.529307
OrderAmountHikeFromlastYear    4.706927
CouponUsed                     4.547069
OrderCount                     4.582593
DaySinceLastOrder              5.452931
dtype: float64


In [4]:
# Summary statistics for numerical columns (min, max, mean, quartiles)
display(df.describe().T)

# Summary statistics for categorical/object columns
display(df.describe(include=['O']).T)

,count,mean,std,min,25%,50%,75%,max
CustomerID,5630.0,52815.500000,1625.385339,50001.0,51408.25,52815.5,54222.75,55630.0
Churn,5630.0,0.168384,0.374240,0.0,0.00,0.0,0.00,1.0
Tenure,5366.0,10.189899,8.557241,0.0,2.00,9.0,16.00,61.0
CityTier,5630.0,1.654707,0.915389,1.0,1.00,1.0,3.00,3.0
WarehouseToHome,5379.0,15.639896,8.531475,5.0,9.00,14.0,20.00,127.0
HourSpendOnApp,5375.0,2.931535,0.721926,0.0,2.00,3.0,3.00,5.0
NumberOfDeviceRegistered,5630.0,3.688988,1.023999,1.0,3.00,4.0,4.00,6.0
SatisfactionScore,5630.0,3.066785,1.380194,1.0,2.00,3.0,4.00,5.0
NumberOfAddress,5630.0,4.214032,2.583586,1.0,2.00,3.0,6.00,22.0
Complain,5630.0,0.284902,0.451408,0.0,0.00,0.0,1.00,1.0


C:\Users\Kartikeya\AppData\Local\Temp\ipykernel_7784\3016382176.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df.describe(include=['O']).T)


,count,unique,top,freq
PreferredLoginDevice,5630,3,Mobile Phone,2765
PreferredPaymentMode,5630,7,Debit Card,2314
Gender,5630,2,Male,3384
PreferedOrderCat,5630,6,Laptop & Accessory,2050
MaritalStatus,5630,3,Married,2986


In [5]:
# Inspect target variable class distribution (e.g., 'Churn')
print("Target Class Counts:\n", df['Churn'].value_counts())
print("\nTarget Class Percentages:\n", df['Churn'].value_counts(normalize=True) * 100)

# Inspect unique categories in object columns to spot typos (e.g., 'CC' vs 'Credit Card')
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

for col in categorical_cols:
    print(f"\nUnique values in '{col}':")
    print(df[col].value_counts())

Target Class Counts:
 Churn
0    4682
1     948
Name: count, dtype: int64

Target Class Percentages:
 Churn
0    83.161634
1    16.838366
Name: proportion, dtype: float64

Unique values in 'PreferredLoginDevice':
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

Unique values in 'PreferredPaymentMode':
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

Unique values in 'Gender':
Gender
Male      3384
Female    2246
Name: count, dtype: int64

Unique values in 'PreferedOrderCat':
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64

Unique values in 'MaritalStatus':
MaritalStatus
Married     2986
Single      1

C:\Users\Kartikeya\AppData\Local\Temp\ipykernel_7784\651786164.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object', 'category']).columns


In [6]:
# Comprehensive dictionary for category mapping
category_replacements = {
    'PreferredLoginDevice': {
        'Phone': 'Mobile Phone'
    },
    'PreferredPaymentMode': {
        'CC': 'Credit Card',
        'COD': 'Cash on Delivery'
    },
    'PreferedOrderCat': {
        'Mobile': 'Mobile Phone'
    }
}

# Apply replacements
df = df.replace(category_replacements)

# Quick verification check
for col in category_replacements.keys():
    print(f"\n--- Cleaned '{col}' ---")
    print(df[col].value_counts())


--- Cleaned 'PreferredLoginDevice' ---
PreferredLoginDevice
Mobile Phone    3996
Computer        1634
Name: count, dtype: int64

--- Cleaned 'PreferredPaymentMode' ---
PreferredPaymentMode
Debit Card          2314
Credit Card         1774
E wallet             614
Cash on Delivery     514
UPI                  414
Name: count, dtype: int64

--- Cleaned 'PreferedOrderCat' ---
PreferedOrderCat
Mobile Phone          2080
Laptop & Accessory    2050
Fashion                826
Grocery                410
Others                 264
Name: count, dtype: int64


In [7]:
# Drop unique IDs if present
if 'CustomerID' in df.columns:
    df = df.drop(columns=['CustomerID'])

# Separate features (X) and target (y)
X = df.drop(columns=['Churn'])
y = df['Churn']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (5630, 18)
Target shape: (5630,)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

Training set size: (4504, 18)
Testing set size: (1126, 18)


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define feature types
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Define numerical pipeline (Impute missing with median -> Scale)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Define categorical pipeline (Impute missing -> One-Hot Encode)
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into a single ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit transformer on training set and transform both sets
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("Preprocessed X_train shape:", X_train_prepared.shape)
print("Preprocessed X_test shape:", X_test_prepared.shape)

Preprocessed X_train shape: (4504, 30)
Preprocessed X_test shape: (1126, 30)


C:\Users\Kartikeya\AppData\Local\Temp\ipykernel_7784\84903947.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


In [10]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# 1. Setup Stratified 5-Fold Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Instantiate Models
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Set scale_pos_weight to ~4.8 to account for the 83/17 churn imbalance
xgb_model = XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    scale_pos_weight=4.8, 
    random_state=42, 
    eval_metric='logloss'
)

# 3. Evaluate Random Forest with 5-Fold CV
rf_results = cross_validate(
    rf_model, X_train_prepared, y_train, cv=cv, scoring=['f1', 'roc_auc']
)

# 4. Evaluate XGBoost with 5-Fold CV
xgb_results = cross_validate(
    xgb_model, X_train_prepared, y_train, cv=cv, scoring=['f1', 'roc_auc']
)

# 5. Display Performance Summary
print("--- Random Forest (5-Fold CV) ---")
print("Mean F1-Score: ", rf_results['test_f1'].mean())
print("Mean ROC-AUC:  ", rf_results['test_roc_auc'].mean())

print("\n--- XGBoost (5-Fold CV) ---")
print("Mean F1-Score: ", xgb_results['test_f1'].mean())
print("Mean ROC-AUC:  ", xgb_results['test_roc_auc'].mean())

--- Random Forest (5-Fold CV) ---
Mean F1-Score:  0.8329335504732507
Mean ROC-AUC:   0.9784586804170792

--- XGBoost (5-Fold CV) ---
Mean F1-Score:  0.8376764631791469
Mean ROC-AUC:   0.9681045566816714


In [12]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# Define the hyperparameter search grid
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0],
    'scale_pos_weight': [4.8]  # Retaining our class imbalance weight
}

# Instantiate base model
xgb = XGBClassifier(random_state=42, eval_metric='logloss')

# Enable return_train_score to evaluate training error alongside dev error
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    return_train_score=True,
    n_jobs=-1
)

grid_search.fit(X_train_prepared, y_train)

# Retrieve scores for the optimal hyperparameter combination
best_idx = grid_search.best_index_
train_f1 = grid_search.cv_results_['mean_train_score'][best_idx]
dev_f1 = grid_search.cv_results_['mean_test_score'][best_idx]

print(f"Mean Training F1-Score: {train_f1:.4f}")
print(f"Mean Dev (CV) F1-Score:   {dev_f1:.4f}")
print(f"Overfitting Gap:        {train_f1 - dev_f1:.4f}")

# Save the best estimator for final testing
best_xgb_model = grid_search.best_estimator_

Mean Training F1-Score: 1.0000
Mean Dev (CV) F1-Score:   0.8809
Overfitting Gap:        0.1191


In [13]:
from sklearn.metrics import f1_score

# Evaluate best estimator directly on training and test sets
y_train_pred = best_xgb_model.predict(X_train_prepared)
y_test_pred = best_xgb_model.predict(X_test_prepared)

print("Training F1-Score:", f1_score(y_train, y_train_pred))
print("Test F1-Score:    ", f1_score(y_test, y_test_pred))

Training F1-Score: 1.0
Test F1-Score:     0.9736842105263158


In [14]:
from sklearn.metrics import classification_report, confusion_matrix

print("=== Classification Report ===")
print(classification_report(y_test, y_test_pred, target_names=['Non-Churn', 'Churn']))

print("\n=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_test_pred)
print(f"True Negatives  (Loyal predicted correctly): {cm[0][0]}")
print(f"False Positives (Loyal predicted as Churn): {cm[0][1]}")
print(f"False Negatives (Churn predicted as Loyal): {cm[1][0]}")
print(f"True Positives  (Churn predicted correctly): {cm[1][1]}")

=== Classification Report ===
              precision    recall  f1-score   support

   Non-Churn       0.99      0.99      0.99       936
       Churn       0.97      0.97      0.97       190

    accuracy                           0.99      1126
   macro avg       0.98      0.98      0.98      1126
weighted avg       0.99      0.99      0.99      1126


=== Confusion Matrix ===
True Negatives  (Loyal predicted correctly): 931
False Positives (Loyal predicted as Churn): 5
False Negatives (Churn predicted as Loyal): 5
True Positives  (Churn predicted correctly): 185


In [16]:
# Check feature importances to see if one or two features dominate 95%+ of the predictions
import pandas as pd
import matplotlib.pyplot as plt

# Extract feature names after ColumnTransformer preprocessing
cat_encoder = preprocessor.named_transformers_['cat']['encoder']
cat_feature_names = cat_encoder.get_feature_names_out(cat_cols).tolist()
all_feature_names = num_cols + cat_feature_names

# Get feature importances from XGBoost
importances = best_xgb_model.feature_importances_
feature_imp_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("=== Top 10 Most Important Features ===")
print(feature_imp_df.head(10))

=== Top 10 Most Important Features ===
                                Feature  Importance
0                                Tenure    0.134177
7                              Complain    0.075046
22             PreferedOrderCat_Fashion    0.052401
25        PreferedOrderCat_Mobile Phone    0.052096
23             PreferedOrderCat_Grocery    0.051723
1                              CityTier    0.044864
24  PreferedOrderCat_Laptop & Accessory    0.042608
6                       NumberOfAddress    0.037426
5                     SatisfactionScore    0.037194
29                 MaritalStatus_Single    0.036503


In [17]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# Define a regularized parameter grid to prevent overfitting
param_grid_regularized = {
    'max_depth': [3, 4, 5],                  # Lower tree depth (prevents memorization)
    'learning_rate': [0.03, 0.05, 0.1],
    'n_estimators': [100, 150],
    'subsample': [0.7, 0.8],                  # Sample fraction of rows per tree
    'colsample_bytree': [0.7, 0.8],           # Sample fraction of features per tree
    'gamma': [0.1, 0.5, 1.0],                # Minimum loss reduction required for split
    'scale_pos_weight': [4.8]
}

xgb_reg = XGBClassifier(random_state=42, eval_metric='logloss')

grid_search_reg = GridSearchCV(
    estimator=xgb_reg,
    param_grid=param_grid_regularized,
    cv=5,
    scoring='f1',
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

# Fit on training data
grid_search_reg.fit(X_train_prepared, y_train)

# Evaluate scores
best_idx = grid_search_reg.best_index_
train_f1 = grid_search_reg.cv_results_['mean_train_score'][best_idx]
dev_f1 = grid_search_reg.cv_results_['mean_test_score'][best_idx]

print("\n=== Regularized Model Cross-Validation Results ===")
print("Best Parameters:", grid_search_reg.best_params_)
print(f"Mean Training F1-Score: {train_f1:.4f}")
print(f"Mean Dev (CV) F1-Score:   {dev_f1:.4f}")
print(f"Overfitting Gap:        {train_f1 - dev_f1:.4f}")

# Store regularized estimator
best_xgb_model = grid_search_reg.best_estimator_

Fitting 5 folds for each of 216 candidates, totalling 1080 fits

=== Regularized Model Cross-Validation Results ===
Best Parameters: {'colsample_bytree': 0.8, 'gamma': 0.5, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 150, 'scale_pos_weight': 4.8, 'subsample': 0.7}
Mean Training F1-Score: 0.9565
Mean Dev (CV) F1-Score:   0.8249
Overfitting Gap:        0.1316


In [18]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Predict on the held-out test set using the regularized model
y_pred = best_xgb_model.predict(X_test_prepared)
y_proba = best_xgb_model.predict_proba(X_test_prepared)[:, 1]

# Display final classification metrics
print("=== Final Test Set Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Non-Churn', 'Churn']))
print("Test ROC-AUC Score:", roc_auc_score(y_test, y_proba))

# Display raw confusion matrix counts
cm = confusion_matrix(y_test, y_pred)
print("\n=== Confusion Matrix ===")
print(f"True Negatives  (Loyal predicted correctly): {cm[0][0]}")
print(f"False Positives (Loyal predicted as Churn): {cm[0][1]}")
print(f"False Negatives (Churn predicted as Loyal): {cm[1][0]}")
print(f"True Positives  (Churn predicted correctly): {cm[1][1]}")

=== Final Test Set Classification Report ===
              precision    recall  f1-score   support

   Non-Churn       1.00      0.96      0.98       936
       Churn       0.84      0.99      0.91       190

    accuracy                           0.97      1126
   macro avg       0.92      0.98      0.94      1126
weighted avg       0.97      0.97      0.97      1126

Test ROC-AUC Score: 0.9928081421502473

=== Confusion Matrix ===
True Negatives  (Loyal predicted correctly): 900
False Positives (Loyal predicted as Churn): 36
False Negatives (Churn predicted as Loyal): 2
True Positives  (Churn predicted correctly): 188


In [19]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Get predicted probabilities for class 1 (Churn)
y_proba = best_xgb_model.predict_proba(X_test_prepared)[:, 1]

# Test thresholds from 0.50 to 0.85
print("Threshold | Precision | Recall | F1-Score | False Positives | False Negatives")
print("-" * 75)

for threshold in np.arange(0.50, 0.90, 0.05):
    y_pred_custom = (y_proba >= threshold).astype(int)
    
    prec = precision_score(y_test, y_pred_custom)
    rec = recall_score(y_test, y_pred_custom)
    f1 = f1_score(y_test, y_pred_custom)
    cm = confusion_matrix(y_test, y_pred_custom)
    
    fp = cm[0][1]
    fn = cm[1][0]
    
    print(f"  {threshold:.2f}    |   {prec:.3f}   |  {rec:.3f} |  {f1:.3f}   |       {fp:2d}        |       {fn:2d}")

Threshold | Precision | Recall | F1-Score | False Positives | False Negatives
---------------------------------------------------------------------------
  0.50    |   0.839   |  0.989 |  0.908   |       36        |        2
  0.55    |   0.869   |  0.974 |  0.918   |       28        |        5
  0.60    |   0.883   |  0.953 |  0.916   |       24        |        9
  0.65    |   0.901   |  0.905 |  0.903   |       19        |       18
  0.70    |   0.908   |  0.832 |  0.868   |       16        |       32
  0.75    |   0.910   |  0.795 |  0.848   |       15        |       39
  0.80    |   0.925   |  0.774 |  0.842   |       12        |       43
  0.85    |   0.957   |  0.705 |  0.812   |        6        |       56


In [20]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# 1. Get continuous predicted probabilities for the churn class (1)
y_proba = best_xgb_model.predict_proba(X_test_prepared)[:, 1]

# 2. Apply chosen decision threshold (e.g., 0.60)
OPTIMAL_THRESHOLD = 0.60
y_pred_tuned = (y_proba >= OPTIMAL_THRESHOLD).astype(int)

# 3. Print updated report
print(f"=== Performance at Threshold {OPTIMAL_THRESHOLD} ===")
print(classification_report(y_test, y_pred_tuned, target_names=['Non-Churn', 'Churn']))

cm = confusion_matrix(y_test, y_pred_tuned)
print("=== Confusion Matrix ===")
print(f"False Positives (Wasted Discounts): {cm[0][1]}")
print(f"False Negatives (Missed Churners): {cm[1][0]}")

=== Performance at Threshold 0.6 ===
              precision    recall  f1-score   support

   Non-Churn       0.99      0.97      0.98       936
       Churn       0.88      0.95      0.92       190

    accuracy                           0.97      1126
   macro avg       0.94      0.96      0.95      1126
weighted avg       0.97      0.97      0.97      1126

=== Confusion Matrix ===
False Positives (Wasted Discounts): 24
False Negatives (Missed Churners): 9


In [21]:
import joblib

# 1. Save the preprocessing pipeline
joblib.dump(preprocessor, 'churn_preprocessor.joblib')

# 2. Save the trained XGBoost model
joblib.dump(best_xgb_model, 'churn_xgb_model.joblib')

# 3. Save threshold metadata
metadata = {'optimal_threshold': 0.60}
joblib.dump(metadata, 'model_metadata.joblib')

print("Model, preprocessor, and metadata saved successfully!")

Model, preprocessor, and metadata saved successfully!


In [24]:
import joblib
import pandas as pd

# Load saved artifacts
loaded_preprocessor = joblib.load('churn_preprocessor.joblib')
loaded_model = joblib.load('churn_xgb_model.joblib')
metadata = joblib.load('model_metadata.joblib')

threshold = metadata['optimal_threshold']

# Option A: Grab a real sample row directly from your test set (safest method)
new_raw_data = X_test.iloc[[0]] 

# Option B: Or construct custom input ensuring ALL features from X_train are present
# new_raw_data = pd.DataFrame([X_train.iloc[0].to_dict()])

# Process and predict
X_new_prepared = loaded_preprocessor.transform(new_raw_data)
prob = loaded_model.predict_proba(X_new_prepared)[:, 1][0]
prediction = int(prob >= threshold)

print(f"Churn Probability: {prob:.4f}")
print(f"Predicted Class (Threshold {threshold}): {'Churn' if prediction == 1 else 'Non-Churn'}")

Churn Probability: 0.0236
Predicted Class (Threshold 0.6): Non-Churn
